In [1]:
import pandas as pd
df = pd.read_csv('../datasets/feature_engineered_data/feature_engineered_cars.csv')
# Remove the single instance of 'N' to prevent CV encoding errors
df = df[df['Fuel type'] != 'N'].reset_index(drop = True)
df.head()

,Model year,Make,Vehicle class,Engine size (L),Cylinders,Fuel type,Combined (L/100 km),CO2 emissions (g/km),Transmission type,Gears
0,2015,Acura,Compact,2.0,4,Z,8.3,191,AS,5
1,2015,Acura,Compact,2.4,4,Z,9.3,214,M,6
2,2015,Acura,Compact,1.5,4,Z,6.1,140,AV,7
3,2015,Acura,Sport utility vehicle: Small,3.5,6,Z,11.1,255,AS,6
4,2015,Acura,Sport utility vehicle: Small,3.5,6,Z,10.6,244,AS,6


**Workflow**

                    Entire dataset
                         │
                  Train/Test Split
                    /          \
                   /            \
              80% Training     20% Test
                  │                 │
                  │                 │
              K-Fold CV             │
                  │                 │
            Model selection         │
                  │                 │
            Hyperparameter          │
               tuning               │
                  │                 │
                  └───────┐         │
                          ↓         ↓
                    Final model → FINAL TEST

**Imports**

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_validate,
    GridSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, Ridge, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor


**Splitting the data into training set and test set**

##### Feature selection: exclude Combined (L/100 km)

Combined (L/100 km) is excluded from the model because this project is intended for a **car-manufacturer use case**: the goal is to predict CO₂ emissions from vehicle specifications rather than rely on a fuel-consumption measurement that is itself extremely strongly associated with the target (Pearson ≈ 0.94; Spearman ≈ 0.97 from EDA).

Initial experiments also showed that Combined (L/100 km) dominated model feature importance. Keeping it would therefore make the prediction task easier, but it would also make the model depend heavily on a variable that may not be available at the intended prediction stage.

Engine size (L) and Cylinders are **not removed at this stage**. They are strongly correlated, so we will assess their redundancy with VIF and then compare three feature sets empirically before making the final choice.

In [3]:
# Start with Model A (Engine size + Cylinders) as the full candidate set.
# After running the feature-selection experiment above, change this section
# to Model B or Model C if the results justify a simpler feature set.

X = df.drop(columns = [
    'CO2 emissions (g/km)',
    'Combined (L/100 km)'
])

y = df['CO2 emissions (g/km)']

#### Multicollinearity check using VIF

Variance Inflation Factor (VIF) measures how much the variance of an estimated regression coefficient is inflated because a predictor can be explained by the other predictors.

For VIF, the **intercept/constant is included** in the auxiliary regressions. The target variable is not included, and `Combined (L/100 km)` is already excluded because it is not part of the intended feature set.

In [4]:
vif_cols = [
    'Model year',
    'Engine size (L)',
    'Cylinders',
    'Gears'
]

X_vif = df[vif_cols].dropna().copy()
X_vif = sm.add_constant(X_vif)

vif = pd.DataFrame({
    'Feature': X_vif.columns,
    'VIF': [
        variance_inflation_factor(X_vif.values, i)
        for i in range(X_vif.shape[1])
    ]
})

vif = vif[vif['Feature'] != 'const'].reset_index(drop=True)

vif

,Feature,VIF
0,Model year,1.073081
1,Engine size (L),6.765561
2,Cylinders,6.813822
3,Gears,1.163876


#### How to interpret VIF

VIF is a rule-of-thumb diagnostic rather than a hard cutoff:

- **VIF ≈ 1:** essentially no multicollinearity
- **VIF around 1–5:** generally low/acceptable
- **VIF around 5–10:** potentially problematic and worth investigating
- **VIF > 10:** strong evidence of serious multicollinearity

Because `Engine size (L)` and `Cylinders` are strongly correlated, their VIF values should be examined together. If one is removed, VIF should be recalculated to confirm that the remaining predictors are no longer redundant.

The three feature-set experiment below is therefore still necessary: **VIF identifies redundancy, but it does not by itself establish which of two correlated predictors gives the better predictive trade-off.**

#### Engine size vs. cylinders: feature-selection experiment

The correlation/VIF analysis tells us that `Engine size (L)` and `Cylinders` contain substantial overlapping information, but **VIF alone does not tell us which feature is better to keep**.

Therefore, three candidate feature sets are compared:

- **Model A:** Engine size + Cylinders + all other features
- **Model B:** Engine size + all other features
- **Model C:** Cylinders + all other features

The comparison uses the same train/test split, 5-fold cross-validation, preprocessing approach, and model settings. This keeps the feature-selection decision evidence-based rather than removing a feature solely because its VIF is high.

In [5]:
kf = KFold(
    n_splits = 5,
    shuffle = True,
    random_state = 42
)

In [6]:
# Base features: everything except the target and Combined.
base_features = [
    col for col in df.columns
    if col not in ['CO2 emissions (g/km)', 'Combined (L/100 km)', 'Engine size (L)', 'Cylinders']
]

feature_sets = {
    'Model A - Engine size + Cylinders': base_features + ['Engine size (L)', 'Cylinders'],
    'Model B - Engine size only': base_features + ['Engine size (L)'],
    'Model C - Cylinders only': base_features + ['Cylinders']
}

def make_preprocessors(feature_cols):
    cat_cols = [
        col for col in feature_cols
        if col in ['Make', 'Vehicle class', 'Fuel type', 'Transmission type']
    ]
    num_cols = [
        col for col in feature_cols
        if col not in cat_cols
    ]

    scaled = ColumnTransformer(
        transformers = [
            (
                'cat',
                OneHotEncoder(
                    drop = 'first',
                    handle_unknown = 'ignore'
                ),
                cat_cols
            ),
            (
                'num',
                StandardScaler(),
                num_cols
            )
        ]
    )

    unscaled = ColumnTransformer(
        transformers = [
            (
                'cat',
                OneHotEncoder(
                    drop = 'first',
                    handle_unknown = 'ignore'
                ),
                cat_cols
            )
        ],
        remainder = 'passthrough'
    )

    return scaled, unscaled


def evaluate_feature_set(feature_cols, feature_set_name):
    X_fs = df[feature_cols]
    y_fs = df['CO2 emissions (g/km)']

    X_train_fs, X_test_fs, y_train_fs, y_test_fs = train_test_split(
        X_fs,
        y_fs,
        test_size = 0.2,
        random_state = 42
    )

    scaled_fs, unscaled_fs = make_preprocessors(feature_cols)

    models = {
        'Linear Regression': Pipeline([
            ('preprocessor', scaled_fs),
            ('model', LinearRegression())
        ]),
        'Ridge Regression': Pipeline([
            ('preprocessor', scaled_fs),
            ('model', Ridge(alpha = 1.0))
        ]),
        'Decision Tree': Pipeline([
            ('preprocessor', unscaled_fs),
            ('model', DecisionTreeRegressor(random_state = 42))
        ]),
        'Random Forest': Pipeline([
            ('preprocessor', unscaled_fs),
            ('model', RandomForestRegressor(
                n_estimators = 200,
                random_state = 42,
                n_jobs = 2
            ))
        ])
    }

    rows = []

    for model_name, pipeline in models.items():
        cv_results = cross_validate(
            pipeline,
            X_train_fs,
            y_train_fs,
            cv = kf,
            scoring = {
                'r2': 'r2',
                'rmse': 'neg_root_mean_squared_error',
                'mae': 'neg_mean_absolute_error'
            },
            n_jobs = 2
        )

        rows.append({
            'Feature Set': feature_set_name,
            'Model': model_name,
            'CV R2': cv_results['test_r2'].mean(),
            'CV RMSE': -cv_results['test_rmse'].mean(),
            'CV MAE': -cv_results['test_mae'].mean()
        })

    return rows


feature_selection_results = []

for feature_set_name, feature_cols in feature_sets.items():
    feature_selection_results.extend(
        evaluate_feature_set(feature_cols, feature_set_name)
    )

feature_selection_comparison = (
    pd.DataFrame(feature_selection_results)
    .sort_values(
        ['Model', 'CV R2'],
        ascending = [True, False]
    )
    .reset_index(drop = True)
)

feature_selection_comparison

,Feature Set,Model,CV R2,CV RMSE,CV MAE
0,Model B - Engine size only,Decision Tree,0.916629,17.454819,11.071799
1,Model A - Engine size + Cylinders,Decision Tree,0.913989,17.738081,11.157806
2,Model C - Cylinders only,Decision Tree,0.891037,19.959415,13.055290
3,Model A - Engine size + Cylinders,Linear Regression,0.844664,23.820153,17.774073
4,Model B - Engine size only,Linear Regression,0.840399,24.145114,18.074480
5,Model C - Cylinders only,Linear Regression,0.829166,24.986244,18.532747
6,Model A - Engine size + Cylinders,Random Forest,0.930356,15.966489,10.458536
7,Model B - Engine size only,Random Forest,0.930317,15.970815,10.449216
8,Model C - Cylinders only,Random Forest,0.913044,17.835905,12.172696
9,Model A - Engine size + Cylinders,Ridge Regression,0.844320,23.846405,17.788233


**Finalized X and y**

In [7]:
X = df.drop(columns = [
    'CO2 emissions (g/km)',
    'Combined (L/100 km)',
    'Cylinders'
])

y = df['CO2 emissions (g/km)']

Engine Size was retained and Cylinders was removed because Cylinders provides minimal additional predictive value when Engine Size is already included, while introducing substantial redundancy/multicollinearity. This choice therefore favors a simpler feature set with essentially no meaningful loss in predictive performance.

**Train Test Split**

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42
)

**Defining Categorical and Numerical Columns**

In [9]:
categorical_cols = [
    'Make',
    'Vehicle class',
    'Fuel type',
    'Transmission type'
]

numerical_cols = [
    col for col in X.columns
    if col not in categorical_cols
]

**Preprocessing Pipelines**

In [10]:
# For linear Regression and Ridge Regression
preprocessor_scaled = ColumnTransformer(
    transformers = [
        (
            'cat',
            OneHotEncoder(
                drop = 'first',
                handle_unknown = 'ignore'
            ),
            categorical_cols
        ),
        (
            'num',
            StandardScaler(),
            numerical_cols
        )
    ]
)

# For decision tree and random forest
preprocessor_unscaled = ColumnTransformer(
    transformers = [
        (
            'cat',
            OneHotEncoder(
                drop = 'first',
                handle_unknown = 'ignore'
            ),
            categorical_cols
        )
    ],
    remainder = 'passthrough'
)

**K fold**

In [11]:
kf = KFold(
    n_splits = 5,
    shuffle = True,
    random_state = 42
)

**Evaluation Function**

In [12]:
def evaluate_model(model, X_train, y_train, X_test, y_test):

    # Cross-validation
    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv = kf,
        scoring = {
            'r2': 'r2',
            'rmse': 'neg_root_mean_squared_error',
            'mae': 'neg_mean_absolute_error'
        },
        return_train_score=True
    )

    # Fit on complete training data
    model.fit(X_train, y_train)

    # Test prediction
    y_pred = model.predict(X_test)

    test_r2 = r2_score(y_test, y_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    test_mae = mean_absolute_error(y_test, y_pred)

    print("CV R²:", cv_results['test_r2'].mean())
    print("CV R² Std:", cv_results['test_r2'].std())

    print("\nTest R²:", test_r2)
    print("Test RMSE:", test_rmse)
    print("Test MAE:", test_mae)

    print(
        "\nTrain-CV R²:",
        cv_results['train_r2'].mean()
    )

    return {
        'CV R2': cv_results['test_r2'].mean(),
        'CV R2 Std': cv_results['test_r2'].std(),
        'Test R2': test_r2,
        'Test RMSE': test_rmse,
        'Test MAE': test_mae,
        'Train CV R2': cv_results['train_r2'].mean()
    }

**Linear Regression**

In [13]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor_scaled),
    ('model', LinearRegression())
])

**Evaluation**

In [14]:
lr_results = evaluate_model(
    lr_pipeline,
    X_train,
    y_train,
    X_test,
    y_test
)

CV R²: 0.8403991920414681
CV R² Std: 0.012957096024765986

Test R²: 0.8459619869221139
Test RMSE: 23.554689697873396
Test MAE: 17.839126265140198

Train-CV R²: 0.8440983734258556


**Ridge Regression [Regularization benchmark]**

Ridge is retained as a regularized linear baseline. It should not be presented as a substitute for feature-selection/VIF analysis; it is an additional model for comparison.

In [15]:
ridge_pipeline = Pipeline([
    ('preprocessor', preprocessor_scaled),
    ('model', Ridge(alpha=1.0))
])

**Evaluation**

In [16]:
ridge_results = evaluate_model(
    ridge_pipeline,
    X_train,
    y_train,
    X_test,
    y_test
)

CV R²: 0.839950331638866
CV R² Std: 0.012802967623415672

Test R²: 0.8455696539757798
Test RMSE: 23.58466737594435
Test MAE: 17.87978868283119

Train-CV R²: 0.8435886863658328


**Decision Tree**

In [17]:
dt_pipeline = Pipeline([
    ('preprocessor', preprocessor_unscaled),
    ('model', DecisionTreeRegressor(
        random_state = 42
    ))
])

**Evaluation**

In [18]:
dt_results = evaluate_model(
    dt_pipeline,
    X_train,
    y_train,
    X_test,
    y_test
)

CV R²: 0.9154679795480046
CV R² Std: 0.006395581543578944

Test R²: 0.9357390267074823
Test RMSE: 15.21377828469968
Test MAE: 9.708870542562144

Train-CV R²: 0.9741793570857764


**Random Forest**

In [19]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor_unscaled),
    ('model', RandomForestRegressor(
        n_estimators = 200,
        random_state = 42,
        n_jobs = 2
))
])

**Evaluation**

In [20]:
rf_results = evaluate_model(
    rf_pipeline,
    X_train,
    y_train,
    X_test,
    y_test
)

CV R²: 0.9302883222554794
CV R² Std: 0.003080860101896496

Test R²: 0.9475093343323705
Test RMSE: 13.750056208310472
Test MAE: 9.254202662337002

Train-CV R²: 0.9714779397729624


In [21]:
# Define and evaluate SGD Regressor (Stochastic Gradient Descent / Gradient Descent)
sgd_pipeline = Pipeline([
    ('preprocessor', preprocessor_scaled),  # SGD requires scaling
    ('model', SGDRegressor(max_iter=1000, tol=1e-3, random_state=42))
])

sgd_results = evaluate_model(
    sgd_pipeline,
    X_train,
    y_train,
    X_test,
    y_test
)


D:\CO2_PROJECT_2.0\.venv\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:1620: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


D:\CO2_PROJECT_2.0\.venv\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:1620: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


D:\CO2_PROJECT_2.0\.venv\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:1620: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


D:\CO2_PROJECT_2.0\.venv\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:1620: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


D:\CO2_PROJECT_2.0\.venv\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:1620: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


CV R²: 0.7776273436112403
CV R² Std: 0.013014343763085662

Test R²: 0.7936688053976964
Test RMSE: 27.26125716462721
Test MAE: 20.414090828105515

Train-CV R²: 0.7834118218572351


D:\CO2_PROJECT_2.0\.venv\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:1620: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


In [22]:
# Define and evaluate XGBoost
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor_unscaled),
    ('model', XGBRegressor(
        n_estimators = 200,
        max_depth = 6,
        learning_rate = 0.1,
        random_state = 42,
        n_jobs = 2
    ))
])

xgb_results = evaluate_model(
    xgb_pipeline,
    X_train,
    y_train,
    X_test,
    y_test
)


CV R²: 0.9307641863822937
CV R² Std: 0.004142001699253429

Test R²: 0.9399537444114685
Test RMSE: 14.706404057841604
Test MAE: 10.490777969360352

Train-CV R²: 0.9484435796737671


**Check for overfitting**

In [23]:
rf_pipeline.fit(X_train, y_train)

train_r2 = rf_pipeline.score(X_train, y_train)
test_r2 = rf_pipeline.score(X_test, y_test)

print("Train R²:", train_r2)
print("Test R²:", test_r2)
print("Gap:", train_r2 - test_r2)

Train R²: 0.9695751144589425
Test R²: 0.9475093343323705
Gap: 0.02206578012657201


**Comparison Table**

In [24]:
comparison = pd.DataFrame([
    {
        'Model': 'Linear Regression',
        **lr_results
    },
    {
        'Model': 'Ridge Regression',
        **ridge_results
    },
    {
        'Model': 'SGD Regression (Gradient Descent)',
        **sgd_results
    },
    {
        'Model': 'Decision Tree',
        **dt_results
    },
    {
        'Model': 'Random Forest',
        **rf_results
    },
    {
        'Model': 'XGBoost',
        **xgb_results
    }
])

comparison


,Model,CV R2,CV R2 Std,Test R2,Test RMSE,Test MAE,Train CV R2
0,Linear Regression,0.840399,0.012957,0.845962,23.554690,17.839126,0.844098
1,Ridge Regression,0.839950,0.012803,0.845570,23.584667,17.879789,0.843589
2,SGD Regression (Gradient Descent),0.777627,0.013014,0.793669,27.261257,20.414091,0.783412
3,Decision Tree,0.915468,0.006396,0.935739,15.213778,9.708871,0.974179
4,Random Forest,0.930288,0.003081,0.947509,13.750056,9.254203,0.971478
5,XGBoost,0.930764,0.004142,0.939954,14.706404,10.490778,0.948444


**Hyperparameter Tuning**

(a) Decision Tree

In [25]:
from sklearn.model_selection import RandomizedSearchCV

dt_params = {
    'model__max_depth': [None, 3, 5, 7, 10, 15, 20, 25],
    'model__min_samples_split': [2, 5, 10, 15, 20],
    'model__min_samples_leaf': [1, 2, 5, 10],
    'model__max_features': [None, 'sqrt', 'log2']
}

dt_search = RandomizedSearchCV(
    estimator = dt_pipeline,
    param_distributions = dt_params,
    n_iter = 20,
    cv = kf,
    scoring = 'r2',
    random_state = 42,
    n_jobs = 2
)

dt_search.fit(X_train, y_train)

print("Best parameters:")
print(dt_search.best_params_)

print("Best CV R²:")
print(dt_search.best_score_)

Best parameters:
{'model__min_samples_split': 15, 'model__min_samples_leaf': 1, 'model__max_features': None, 'model__max_depth': 25}
Best CV R²:
0.9247335375274677


(b) Random Forest

In [26]:
rf_params = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [None, 10, 15, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2', None]
}

rf_search = RandomizedSearchCV(
    estimator = rf_pipeline,
    param_distributions = rf_params,
    n_iter = 20,
    cv = kf,
    scoring = 'r2',
    random_state = 42,
    n_jobs = 2,
    verbose = 1
)

rf_search.fit(X_train, y_train)

print("Best parameters:")
print(rf_search.best_params_)

print("Best CV R²:")
print(rf_search.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


Best parameters:
{'model__n_estimators': 200, 'model__min_samples_split': 10, 'model__min_samples_leaf': 1, 'model__max_features': None, 'model__max_depth': 30}
Best CV R²:
0.9343337385230278


**Evaluation**


In [27]:
best_rf = rf_search.best_estimator_

y_pred = best_rf.predict(X_test)

print("Test R²:",
      r2_score(y_test, y_pred))

print("Test RMSE:",
      np.sqrt(mean_squared_error(y_test, y_pred)))

print("Test MAE:",
      mean_absolute_error(y_test, y_pred))

Test R²: 0.9499304097680819
Test RMSE: 13.429209589287595
Test MAE: 9.306417857977364


In [28]:
# XGBoost Tuning
xgb_params = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3, 5, 7, 9],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__subsample': [0.6, 0.8, 1.0],
    'model__colsample_bytree': [0.6, 0.8, 1.0]
}

xgb_search = RandomizedSearchCV(
    estimator = xgb_pipeline,
    param_distributions = xgb_params,
    n_iter = 15,
    cv = kf,
    scoring = 'r2',
    random_state = 42,
    n_jobs = 2,
    verbose = 1
)

xgb_search.fit(X_train, y_train)

print("Best XGBoost parameters:")
print(xgb_search.best_params_)

print("Best XGBoost CV R:")
print(xgb_search.best_score_)

best_xgb = xgb_search.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)

print("Test R:", r2_score(y_test, y_pred_xgb))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_xgb)))
print("Test MAE:", mean_absolute_error(y_test, y_pred_xgb))


Fitting 5 folds for each of 15 candidates, totalling 75 fits


Best XGBoost parameters:
{'model__subsample': 1.0, 'model__n_estimators': 200, 'model__max_depth': 7, 'model__learning_rate': 0.2, 'model__colsample_bytree': 0.8}
Best XGBoost CV R:
0.9382436990737915
Test R: 0.9509456753730774
Test RMSE: 13.292360931409416
Test MAE: 9.097761154174805


**Ridge Regression Tuning**

In [29]:
ridge_params = {
    'model__alpha': [0.01, 0.1, 1, 10, 100, 1000]
}

ridge_search = GridSearchCV(
    estimator = ridge_pipeline,
    param_grid = ridge_params,
    cv = kf,
    scoring = 'r2',
    n_jobs = 2
)

ridge_search.fit(X_train, y_train)

print("Best parameters:")
print(ridge_search.best_params_)

print("Best CV R²:")
print(ridge_search.best_score_)

Best parameters:
{'model__alpha': 0.01}
Best CV R²:
0.8403949688079155


In [30]:
print(df['Fuel type'].value_counts())

Fuel type
X    4756
Z    4739
E     328
D     236
Name: count, dtype: int64


In [31]:
import joblib
import os

# Compare tuned models and select the best one
rf_test_r2 = r2_score(y_test, best_rf.predict(X_test))
xgb_test_r2 = r2_score(y_test, best_xgb.predict(X_test))

print(f"Tuned Random Forest Test R2: {rf_test_r2:.5f}")
print(f"Tuned XGBoost Test R2: {xgb_test_r2:.5f}")

if xgb_test_r2 > rf_test_r2:
    print("XGBoost is selected as the production model.")
    best_model = best_xgb
    model_name = 'XGBoost'
    selected_metrics = {
        'R2': xgb_test_r2,
        'RMSE': np.sqrt(mean_squared_error(y_test, best_xgb.predict(X_test))),
        'MAE': mean_absolute_error(y_test, best_xgb.predict(X_test))
    }
else:
    print("Random Forest is selected as the production model.")
    best_model = best_rf
    model_name = 'Random Forest'
    selected_metrics = {
        'R2': rf_test_r2,
        'RMSE': np.sqrt(mean_squared_error(y_test, best_rf.predict(X_test))),
        'MAE': mean_absolute_error(y_test, best_rf.predict(X_test))
    }

os.makedirs('../model', exist_ok=True)
# We dump to both filenames to preserve compatibilities
joblib.dump(best_model, '../model/co2_production_model.pkl')
joblib.dump(best_model, '../model/co2_rf_model.pkl') # compat fallback
print(f"Saved the best model ({model_name}) successfully! Ready for Streamlit.")


Tuned Random Forest Test R2: 0.94993
Tuned XGBoost Test R2: 0.95095
XGBoost is selected as the production model.
Saved the best model (XGBoost) successfully! Ready for Streamlit.


In [32]:
import joblib

# This goes at the top of your app.py file
loaded_model = joblib.load('../Model/co2_rf_model.pkl')
loaded_model

# Later in the app, you pass the user's data to it:
# prediction = loaded_model.predict(user_input_data)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['Model year','Make','Vehicle class',...,'Fuel type','Transmission type', 'Gears']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis co